## Integrated Gradients

### Integrated Gradients (PyTorch – Core Implementation)

In [ ]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from PIL import Image

# Model
model = models.resnet50(pretrained=True)
model.eval()

# Image
img = Image.open("image.jpg").convert("RGB")
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])
x = transform(img).unsqueeze(0)

# Baseline (black image)
baseline = torch.zeros_like(x)

def integrated_gradients(model, x, baseline, target_class, steps=50):
    scaled_inputs = [
        baseline + (float(i) / steps) * (x - baseline)
        for i in range(steps + 1)
    ]

    grads = []
    for inp in scaled_inputs:
        inp.requires_grad_()
        output = model(inp)
        model.zero_grad()
        output[0, target_class].backward()
        grads.append(inp.grad.detach())

    avg_grads = torch.mean(torch.stack(grads), dim=0)
    ig = (x - baseline) * avg_grads
    return ig

# Forward
output = model(x)
class_idx = output.argmax()

# Integrated Gradients
ig = integrated_gradients(model, x, baseline, class_idx)

# Convert to saliency map
saliency = ig.abs().max(dim=1)[0]

plt.imshow(saliency[0].cpu(), cmap="hot")
plt.axis("off")
plt.title("Integrated Gradients")
plt.show()


### Integrated Gradients Overlay on Image (Recommended Plot)

In [ ]:
import numpy as np

sal = saliency[0].cpu().numpy()
sal = (sal - sal.min()) / (sal.max() - sal.min())

plt.imshow(img.resize((224, 224)))
plt.imshow(sal, cmap="jet", alpha=0.5)
plt.axis("off")
plt.title("Integrated Gradients Overlay")
plt.show()


### Integrated Gradients with Multiple Baselines (More Robust)

In [ ]:
def ig_multi_baseline(model, x, baselines, target_class, steps=50):
    ig_total = 0
    for baseline in baselines:
        ig_total += integrated_gradients(
            model, x, baseline, target_class, steps
        )
    return ig_total / len(baselines)

baselines = [
    torch.zeros_like(x),
    torch.ones_like(x),
    torch.rand_like(x)
]

ig = ig_multi_baseline(model, x, baselines, class_idx)


### Integrated Gradients for Tabular / Regression Data

In [ ]:
import torch.nn as nn

model = nn.Linear(10, 1)
x = torch.randn(1, 10)
baseline = torch.zeros_like(x)

ig = integrated_gradients(model, x, baseline, target_class=0)
print("Feature Attributions:", ig)


### Integrated Gradients (TensorFlow / Keras)

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

model = tf.keras.applications.ResNet50(weights="imagenet")

img = tf.keras.preprocessing.image.load_img(
    "image.jpg", target_size=(224, 224)
)
img = tf.keras.preprocessing.image.img_to_array(img)
img = tf.keras.applications.resnet.preprocess_input(img)
img = tf.convert_to_tensor(img[None, ...])

baseline = tf.zeros_like(img)

@tf.function
def integrated_gradients_tf(model, x, baseline, target_class, steps=50):
    alphas = tf.linspace(0.0, 1.0, steps)
    ig = tf.zeros_like(x)

    for alpha in alphas:
        with tf.GradientTape() as tape:
            interpolated = baseline + alpha * (x - baseline)
            tape.watch(interpolated)
            preds = model(interpolated)
            loss = preds[:, target_class]
        grads = tape.gradient(loss, interpolated)
        ig += grads

    ig = ig / steps
    return (x - baseline) * ig

preds = model(img)
class_idx = tf.argmax(preds[0])

ig = integrated_gradients_tf(model, img, baseline, class_idx)
saliency = tf.reduce_max(tf.abs(ig), axis=-1)

plt.imshow(saliency[0], cmap="hot")
plt.axis("off")
plt.title("Integrated Gradients (TensorFlow)")
plt.show()
